# PC-Relate kinship on the GRM panel, allele frequencies from the covariate PCs

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread (not OMP: it caps bigsnpr too)
    os.environ[v] = "1"
os.environ.pop("OMP_NUM_THREADS", None)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

## Functions

In [ ]:
def hexbin_safe(ax, x, y, **kw):
    """Pair density; cells with fewer than 21 pairs are not drawn."""
    return ax.hexbin(x, y, **{"gridsize": 120, "bins": "log", "cmap": "Greys", **kw, "mincnt": 21})

def read_tri(path, n):
    """Lower triangle with diagonal, row by row (GCTA .grm.bin order)."""
    tri = np.fromfile(path, dtype="float32")
    assert len(tri) == n * (n + 1) // 2, (len(tri), n)
    return tri

def tri_pairs(n, i, j):
    """Index of pair (i, j), i >= j, in that triangle."""
    i, j = np.maximum(i, j), np.minimum(i, j)
    return i * (i + 1) // 2 + j

## Kinship (R: individual allele frequencies by regression on the PCs, then one pass over SNP blocks)

In [ ]:
STAGE = "pcrelate"
require_ready(cfg, STAGE)
rep, OUT = Report(cfg, STAGE), single_target(cfg, STAGE)
PANEL = single_target(cfg, "grm_panel")
RUN = needs_run(OUT, FORCE)
setenv(RUN=int(RUN), BED=PANEL, PCS=single_target(cfg, "covariate_pcs"), K=cfg.param("pcr_n_pcs"),
       OUT=OUT, THREADS=THREADS)

In [ ]:
%%script Rscript -
suppressMessages({ library(bigsnpr); library(data.table) })
if (Sys.getenv("RUN") != "1") { cat("exists, keeping it (FORCE = True to remake)\n"); quit(save = "no") }
env <- function(k) { v <- Sys.getenv(k); if (!nzchar(v)) stop("missing ", k); v }
bigparallelr::set_blas_ncores(as.integer(env("THREADS")))   # all the work is BLAS here
cat("BLAS threads", bigparallelr::get_blas_ncores(), "\n")

rds <- snp_readBed2(env("BED"), backingfile = tempfile(tmpdir = Sys.getenv("TMPDIR", tempdir())),
                    ncores = as.integer(env("THREADS")))
obj <- snp_attach(rds)
ids <- as.character(obj$fam$sample.ID)
pcs <- fread(env("PCS"), colClasses = list(character = "person_id"))[match(ids, person_id)]
stopifnot(!anyNA(pcs$person_id))
K <- as.integer(env("K"))
X <- cbind(1, as.matrix(pcs[, paste0("PC", seq_len(K)), with = FALSE]))
H <- solve(crossprod(X), t(X))   # (X'X)^-1 X'
n <- nrow(X); m <- ncol(obj$genotypes)
num <- matrix(0, n, n); den <- matrix(0, n, n)
blocks <- split(seq_len(m), ceiling(seq_len(m) / 5000))
for (b in seq_along(blocks)) {
  G <- obj$genotypes[, blocks[[b]]]                   # n x block, NA = missing
  obs <- !is.na(G)
  G0 <- G; G0[!obs] <- colMeans(G, na.rm = TRUE)[col(G)[!obs]]
  mu <- X %*% (H %*% G0) / 2                          # individual allele frequency
  mu <- pmin(pmax(mu, 0.005), 0.995)
  R <- ifelse(obs, G - 2 * mu, 0)
  S <- ifelse(obs, sqrt(mu * (1 - mu)), 0)
  num <- num + tcrossprod(R); den <- den + tcrossprod(S)
  if (b %% 20 == 0) cat("block", b, "of", length(blocks), format(Sys.time(), "%T"), "\n")
}
kin <- num / (4 * den)                                # kinship; diagonal = (1 + F) / 2
dir.create(dirname(env("OUT")), recursive = TRUE, showWarnings = FALSE)
writeBin(kin[upper.tri(kin, diag = TRUE)], env("OUT"), size = 4)   # symmetric: = lower triangle row by row
fwrite(data.table(id = ids), sub("\\.bin$", ".id", env("OUT")), col.names = FALSE)
jsonlite::write_json(list(n = n, n_snps = m, n_pcs = K), sub("\\.bin$", "_meta.json", env("OUT")), auto_unbox = TRUE)
unlink(c(rds, sub("\\.rds$", ".bk", rds)))
cat("done", n, "people,", m, "SNPs\n")

In [ ]:
if RUN:
    mark_done(STAGE, OUT, tools={"bigsnpr": "snp_readBed2", "method": "PC-Relate (Conomos 2016), own implementation"})

## Against the GRM and AoU's kinship

In [ ]:
meta = json.load(open(OUT.replace(".bin", "_meta.json")))
ids = pd.read_csv(OUT.replace(".bin", ".id"), header=None, dtype=str)[0]
n = len(ids)
kin = read_tri(OUT, n)
grm_id = pd.read_csv(target_items(cfg, "grm_shards")[0][0].split(".grm.bin.")[0] + ".grm.id", sep="\t", header=None, dtype=str)[1]
rep.check("PC-Relate and GRM have the same people in the same order", list(grm_id) == list(ids))
grm = np.concatenate([np.fromfile(u, dtype="float32") for u, _ in target_items(cfg, "grm_shards")])
diag = np.arange(1, n + 1) * (np.arange(1, n + 1) + 1) // 2 - 1
off = np.ones(len(kin), bool); off[diag] = False
rep.note(n_snps=meta["n_snps"], n_pcs=meta["n_pcs"], self_kinship_x2_mean=round(float(2 * kin[diag].mean()), 4),
         offdiag_kinship_mean=round(float(kin[off].mean()), 5), grm_offdiag_mean=round(float(grm[off].mean()), 5))
rep.count("n", n)
rep.check("mean off-diagonal kinship ~ 0 (ancestry removed)", abs(kin[off].mean()) < 0.005, f"{kin[off].mean():.5f}")
rep.check("self-kinship x 2 ~ 1", abs(2 * kin[diag].mean() - 1) < 0.05, f"{2 * kin[diag].mean():.3f}")

rel = pd.read_csv(cfg.input("relatedness"), sep="\t", dtype={"i.s": str, "j.s": str})
pos = pd.Series(np.arange(n), index=ids)
rel = rel[rel["i.s"].isin(pos.index) & rel["j.s"].isin(pos.index)]
rel = rel.assign(pcrelate=kin[tri_pairs(n, pos[rel["i.s"]].to_numpy(), pos[rel["j.s"]].to_numpy())])
deg = lambda k: pd.cut(k, [-1, 0.0442, 0.0884, 0.177, 0.354, 1], labels=["unrelated", "3rd", "2nd", "1st", "dup/MZ"])
agree = pd.crosstab(deg(rel["kin"]).rename("aou"), deg(rel["pcrelate"]).rename("pcrelate"))
rep.table(agree.reset_index(), "degree_vs_aou", counts=list(agree.columns))
if len(rel):
    rep.note(aou_pairs=len(rel), corr_with_aou=round(float(np.corrcoef(rel["kin"], rel["pcrelate"])[0, 1]), 3))
display(agree)

In [ ]:
rng = np.random.default_rng(1)
sub = rng.choice(np.flatnonzero(off), size=min(off.sum(), 5_000_000), replace=False)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
hexbin_safe(axes[0], grm[sub] / 2, kin[sub], gridsize=80)
axes[0].plot([-0.1, 0.5], [-0.1, 0.5], "r:", lw=0.8); axes[0].set(xlabel="GRM / 2", ylabel="PC-Relate kinship", title="pairs (random 5M)")
for ax, (lo, hi) in zip(axes[1:], [(-0.05, 0.05), (0.02, 0.6)]):
    for v, lab in [(grm[off] / 2, "GRM / 2"), (kin[off], "PC-Relate")]:
        h, e = np.histogram(v[(v > lo) & (v < hi)], bins=120)
        ax.stairs(np.where(h >= 21, h, 0), e, label=lab)   # bins under 21 pairs not drawn
    if hi > 0.1:
        ax.set_yscale("log")
    ax.set(xlabel="kinship", title=f"{lo} to {hi} (bins with 21+ pairs)")
axes[1].legend(fontsize=8)
rep.fig(fig, "pcrelate_vs_grm", "off-diagonal pairs; hexagons with 21+ pairs"); plt.show()
rep.save()